In [ ]:
from embedder import Embedder

embedder = Embedder()

v = embedder.encode("This is a test sentence")
len(v), v[:5]

In [3]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

v[0]

np.float64(-0.02058203437252893)

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [6]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [7]:
target_doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

len(target_doc["content"])

7219

In [9]:
query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

In [10]:
doc_vector = embedder.encode(target_doc["content"])

similarity = query_vector @ doc_vector
similarity

np.float64(0.36107027225589694)

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [12]:
import numpy as np

texts = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(texts)

X = np.array(X)
X.shape

(295, 384)

In [13]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

In [14]:
scores = X.dot(v)

best_idx = scores.argmax()

best_idx, scores[best_idx]

(np.int64(94), np.float64(0.6489017718578813))

In [15]:
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [16]:
from minsearch import VectorSearch

In [19]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])

index.fit(X, chunks)

In [20]:
query_q4 = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(query_q4)

results = index.search(q4_vector, num_results=5)

results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [21]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [22]:
query_q5 = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(query_q5, num_results=5)

text_files = [r["filename"] for r in text_results]
text_files

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [23]:
q5_vector = embedder.encode(query_q5)

vector_results = index.search(q5_vector, num_results=5)

vector_files = [r["filename"] for r in vector_results]
vector_files

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [24]:
set(vector_files) - set(text_files)

{'02-vector-search/lessons/08-pgvector.md'}

In [25]:
query_q6 = "How do I give the model access to tools?"

q6_vector = embedder.encode(query_q6)

vector_results = index.search(q6_vector, num_results=5)

In [26]:
text_results = text_index.search(query_q6, num_results=5)

In [27]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

In [28]:
results = rrf([vector_results, text_results])

results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'